### Config

In [108]:
import ee
import geemap
import os

try:
  import rasterio
except:
  !pip install rasterio
  import rasterio

try:
  import geopandas as gpd
except:
  !pip install geopandas
  import geopandas as gpd

try:
  import shapely
except:
  !pip install shapely
  import shapely


In [109]:
ee.Initialize(project='geemap-496017')

Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [110]:
#Config
startDate = '2020-01-01'
endDate = '2021-01-01'

# NDVI reference period (uses Landsat 5 + Landsat 8, since Landsat 8 alone does not cover 2000-2013)
refStartDate = '2000-01-01'
refEndDate = '2016-01-01'  # exclusive upper bound -> covers 2000-2015

site_name = 'Macedonia'
year = '2020'


In [111]:
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union

# Input raster
raster_path = "/Users/gregorygiuliani/Downloads/LE_D26_macedonia_2025_maesL2.tif"

# Read the raster's valid-data mask (True where pixels are not nodata) and
# vectorize it, so the AOI matches the raster's exact shape rather than its
# bounding box (relevant if the raster footprint isn't a simple rectangle).
with rasterio.open(raster_path) as src:
    crs = src.crs
    transform = src.transform
    valid_mask = src.read_masks(1) > 0

    geoms = [
        shape(geom)
        for geom, val in shapes(valid_mask.astype(np.uint8), mask=valid_mask, transform=transform)
    ]

# Dissolve all valid-data polygons into a single (multi)polygon
aoi_geom = unary_union(geoms)

aoi = gpd.GeoDataFrame(geometry=[aoi_geom])

# Assign the raster CRS manually
aoi = aoi.set_crs("EPSG:2100")   # Replace with the correct EPSG

# Reproject to WGS84 for display in geemap
aoi = aoi.to_crs(epsg=4326)

Map.add_gdf(
    aoi,
    layer_name="AOI",
    style={"color": "red", "fillOpacity": 0}
)

# Zoom to AOI
xmin, ymin, xmax, ymax = aoi.total_bounds
Map.fit_bounds([[ymin, xmin], [ymax, xmax]])

# Display map
#Map


### Landsat True Color

In [112]:
#Landsat
# Convert GeoPandas -> Earth Engine FeatureCollection
aoi_ee = geemap.gdf_to_ee(aoi)

# Use the AOI's bounding box for spatial filtering (fast), and reserve the exact
# pixel-accurate shape for .clip() below, which is what determines the crop.
aoi_bounds = aoi_ee.geometry().bounds()

#Cloud masking
def cloudMask(image):
  cloudShadowBitmask = (1 << 3)
  cloudBitmask = (1 << 5)
  qa = image.select('QA_PIXEL')
  mask = qa.bitwiseAnd(cloudShadowBitmask).eq(0) \
                .And(qa.bitwiseAnd(cloudBitmask).eq(0))
  return image.updateMask(mask)

#Import Landsat image collection
CollectionLS = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
              .filterBounds(aoi_bounds) \
              .filterDate(startDate, endDate) \
              .filter(ee.Filter.calendarRange(1, 12, 'month')) \
              .map(cloudMask)

bands = ['SR_B4', 'SR_B3', 'SR_B2']

visualizationLS = {
  'bands': bands,
  'min': 7000,
  'max': 14000,
}

#Visualize RGB median
MedianLS = CollectionLS.median().clip(aoi_ee.geometry())
Map.addLayer(MedianLS, visualizationLS, 'Landsat | True Color')


### NDVI reference

In [113]:
def cloudMask(image):
  cloudShadowBitmask = (1 << 3)
  cloudBitmask = (1 << 5)
  qa = image.select('QA_PIXEL')
  mask = qa.bitwiseAnd(cloudShadowBitmask).eq(0) \
                .And(qa.bitwiseAnd(cloudBitmask).eq(0))
  return image.updateMask(mask)

# Landsat 5 and Landsat 8 have different band numbering, so we rename
# the Red and NIR bands to a common naming scheme before merging.
def renameL5(image):
  return image.select(['SR_B3', 'SR_B4'], ['Red', 'NIR'])

def renameL8(image):
  return image.select(['SR_B4', 'SR_B5'], ['Red', 'NIR'])

# Landsat 5 covers the earlier part of the reference period (through ~2012)
CollectionL5_ref = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
              .filterBounds(aoi_bounds) \
              .filterDate(refStartDate, refEndDate) \
              .filter(ee.Filter.calendarRange(1, 12, 'month')) \
              .map(cloudMask) \
              .map(renameL5)

# Landsat 8 covers the later part of the reference period (from 2013)
CollectionL8_ref = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
              .filterBounds(aoi_bounds) \
              .filterDate(refStartDate, refEndDate) \
              .filter(ee.Filter.calendarRange(1, 12, 'month')) \
              .map(cloudMask) \
              .map(renameL8)

# Merge both sensors into a single collection for the 2000-2015 reference period
CollectionLS_ref = CollectionL5_ref.merge(CollectionL8_ref)

MedianLS_ref = CollectionLS_ref.median().clip(aoi_ee.geometry())

#Normalized Difference Vegetation Index (common band names: Red, NIR)
ndvi_ref = MedianLS_ref.normalizedDifference(['NIR', 'Red']).rename('NDVI')

# Fixed display stretch instead of a computed min/max: avoids a synchronous
# reduceRegion().getInfo() call over the full 2000-2015 composite, which was
# timing out. NDVI is bounded [-1, 1]; 0-0.8 gives good contrast for vegetation.
vis_params = {
    'min': 0,
    'max': 0.8,
    'palette': ['red', 'white', 'green']
}

#Visualize
Map.addLayer(ndvi_ref, vis_params, 'Landsat | NDVI Reference period')


In [114]:
# Export to Google Drive (avoids local PROJ/rasterio issues entirely, since this
# submits a server-side Earth Engine task rather than downloading/writing the
# GeoTIFF locally). Files will appear in the 'site_name' folder in your Drive.
geemap.ee_export_image_to_drive(
  image=ndvi_ref,
  description='NDVI_ref_'+site_name+'_Layer',
  fileNamePrefix='NDVI_ref_'+site_name+'_Layer',
  folder=site_name,
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:2100',
  maxPixels=1e10
)


### NDVI Current

In [66]:
#Normalized Difference Vegetation Index
ndvi = MedianLS.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# Fixed display stretch (see NDVI reference cell for rationale)
vis_params = {
    'min': 0,
    'max': 0.8,
    'palette': ['red', 'white', 'green']
}

#Visualize
Map.addLayer(ndvi, vis_params, 'Landsat | NDVI Current')


In [ ]:
# Export to Google Drive
geemap.ee_export_image_to_drive(
  image=ndvi,
  description='NDVI_current_'+site_name+'_'+year+'_Layer',
  fileNamePrefix='NDVI_current_'+site_name+'_'+year,
  folder=site_name,
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:2100',
  maxPixels=1e10
)


### Canopy Cover

In [68]:
# Load Hansen Global Forest Change dataset
gfc = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")

# Select tree cover percentage band
tree_cover = gfc.select("treecover2000").clip(aoi_ee.geometry())

# Visualization parameters
vis_params = {
    "min": 0,
    "max": 100,
    "palette": [
        "ffffff",  # 0%
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837"   # 100%
    ]
}

# Add tree cover layer
Map.addLayer(tree_cover, vis_params, "Tree Cover (%)")

In [ ]:
# Export to Google Drive
geemap.ee_export_image_to_drive(
  image=tree_cover,
  description='Tree_Cover_'+site_name+'_2000_Layer',
  fileNamePrefix='Tree_Cover_'+site_name+'_2000',
  folder=site_name,
  region=aoi_ee.geometry(),
  scale=30,
  crs='EPSG:2100',
  maxPixels=1e10
)
